# Autograd

If a tensor has `requires_grad=True`, PyTorch records operations into a graph. Calling `.backward()` on a **scalar** fills `.grad` on every leaf that requires grad.

$$
y = wx + b,\quad L = y^2
$$

Then $\partial L / \partial w = 2y \cdot x$, and so on.


In [ ]:
import torch

torch.manual_seed(0)


## 1. A first backward pass


In [ ]:
x = torch.tensor([2.0, 4.0, 6.0])
w = torch.tensor([3.0], requires_grad=True)

y = w * x
loss = y.mean()
loss.backward()

print("x.grad (x is a leaf without requires_grad):", x.grad)
print("w.grad:", w.grad)
# d(mean(w*x))/dw = mean(x) = 4


## 2. Gradients **accumulate**

A second `.backward()` **adds** to the existing `.grad`. Call `grad.zero_()` between steps during training. `retain_graph=True` keeps the graph if we need several backwards on the same graph.


In [ ]:
x = torch.tensor(5.0, requires_grad=True)
s = x * x
s.backward(retain_graph=True)
print("after s = x**2, x.grad =", x.grad)  # 2x = 10

m = x * x * x
m.backward(retain_graph=True)
print("after extra x**3, x.grad =", x.grad)  # 10 + 3 * x**2 = 85


## 3. A small linear model, with the chain rule written out


In [ ]:
x = torch.tensor(2.0, requires_grad=True)
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)

y = w * x + b  # 7
loss = y * y   # 49

print("before backward:", x.grad, w.grad, b.grad)
loss.backward()
print("dL/dx = 2y w =", x.grad)  # 42
print("dL/dw = 2y x =", w.grad)  # 28
print("dL/db = 2y     =", b.grad)  # 14


## 4. `torch.no_grad()`

Inference and data preprocessing should not build a graph. Inside `no_grad()`, results have `requires_grad=False` even if inputs do.


In [ ]:
x = torch.tensor(5.0, requires_grad=True)

with torch.no_grad():
    print("inside no_grad, x.requires_grad =", x.requires_grad)
    hidden = x * 2
    print("hidden.requires_grad =", hidden.requires_grad)

visible = x * 3
print("outside, visible.requires_grad =", visible.requires_grad)


## 5. `detach()` shares storage

`c = a.detach()` is a tensor that:

- does **not** require grad
- still **points at the same data** as `a`

Editing `c` (under `no_grad`) changes `a`. We use `a.detach().clone()` when we need a snapshot.


In [ ]:
a = torch.tensor(5.0, requires_grad=True)
b = a * a
c = a.detach()
d = c * 3

print("a.requires_grad:", a.requires_grad, "| c.requires_grad:", c.requires_grad)

loss1 = b.mean()
loss1.backward()
print("dL1/da (from a*a):", a.grad)

a.grad.zero_()
loss2 = (a + d).mean()
loss2.backward()
print("dL2/da (path through d is detached):", a.grad)  # 1.0

with torch.no_grad():
    c.data.fill_(10.0)
print("after editing c, a =", a)
